<a href="https://colab.research.google.com/github/marziqhaqkim-dotcom/ai_course_exercises/blob/main/enriched_service_alerts_2024_29Sep2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# -------------------------------------------------------------------
# 1. Load route metadata and find the exact route_id for Line 4
# -------------------------------------------------------------------
# Update this path to the correct location of your 'routes.csv' in Google Drive
routes_df = pd.read_csv('/content/drive/MyDrive/routes.csv', dtype=str, delimiter=';')

# Filter for Line 4 using route_short_name
line4_info = routes_df[routes_df['route_short_name'] == '4']
line4_route_ids = line4_info['route_id'].unique()

print(f"Mapped Route IDs for Line 4: {line4_route_ids}")

# -------------------------------------------------------------------
# 2. Load Service Alerts Parquet file
# -------------------------------------------------------------------
alerts_df = pd.read_parquet('/content/drive/MyDrive/service_alerts_2024.parquet')

# Ensure route_id is stored as string to avoid type mismatches
alerts_df['route_id'] = alerts_df['route_id'].astype(str)

# -------------------------------------------------------------------
# 3. Extract Cause IDs specifically for Line 4
# -------------------------------------------------------------------
line4_alerts = alerts_df[alerts_df['route_id'].isin(line4_route_ids)].copy()

# Count occurrences of each Cause ID (e.g. 2, 3, 6, 9, 11, 12)
line4_cause_counts = line4_alerts['cause'].value_counts().reset_index()
line4_cause_counts.columns = ['Cause_ID', 'Count']

print("\n--- Line 4 Cause ID Breakdown ---")
print(line4_cause_counts)

# -------------------------------------------------------------------
# 4. Merge full Service Alerts with route.csv for network-wide comparison
# -------------------------------------------------------------------
# Merge alerts with routes on route_id
merged_df = alerts_df.merge(
    routes_df[['route_id', 'route_short_name', 'route_long_name']],
    on='route_id',
    how='left'
)

# Filter for key disruption causes (e.g. 2, 3, 6, 9, 11, 12)
target_causes = [1, 2, 3] # Assuming 1-3 means 1, 2, and 3 based on context
filtered_df = merged_df[merged_df['cause'].isin(target_causes)]

# Group by route_short_name and cause to rank routes by alert frequency
route_summary = (
    filtered_df.groupby(['route_short_name', 'cause'])
    .size()
    .unstack(fill_value=0)
)

print("\n--- Top Routes by Disruption Cause ID ---")
print(route_summary.sum(axis=1).sort_values(ascending=False).head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mapped Route IDs for Line 4: ['9011001000400000' '9011008000400000']

--- Line 4 Cause ID Breakdown ---
   Cause_ID  Count
0         2      6
1         3      3
2        11      1

--- Top Routes by Disruption Cause ID ---
route_short_name
7       3415
21      1455
639     1246
574     1083
626      912
620      729
620X     727
27S      676
27       676
621      541
dtype: int64


In [3]:
import os

# List all files and directories in your Google Drive's MyDrive folder
print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', '.ipynb_checkpoints', 'service_alerts_2024.parquet', 'Module 6', 'Assignment 4', 'Google Colab', 'routes.csv']


In [10]:
import pandas as pd

# -------------------------------------------------------------------
# 1. Load GTFS Stops Data (stops.csv / stops.txt)
# -------------------------------------------------------------------
# Load GTFS stops table to locate Stop Area IDs for Line 4 stations
stops_df = pd.read_csv('/content/drive/MyDrive/stops.csv', dtype=str, delimiter=';')  # Or stops.csv depending on your file format

# List of major station names along Line 4 (Radiohuset -> Gullmarsplan)
line4_station_names = [
    'Radiohuset', 'Karlaplan', 'Odenplan', 'S:t Eriksplan',
    'Fridhemsplan', 'Västerbroplan', 'Hornstull', 'Gullmarsplan'
]

# Search for matching stop_ids in GTFS stops data
# (Matches stop_name against our station list)
pattern = '|'.join(line4_station_names)
line4_stops_info = stops_df[stops_df['stop_name'].str.contains(pattern, case=False, na=False)]

# Extract unique stop_ids (as strings)
line4_stop_ids = line4_stops_info['stop_id'].unique()
print(f"Found {len(line4_stop_ids)} stop IDs matching Line 4 stations.")

# -------------------------------------------------------------------
# 2. Load Service Alerts Parquet Dataset
# -------------------------------------------------------------------
alerts_df = pd.read_parquet('/content/drive/MyDrive/service_alerts_2024.parquet')

# Convert stop_id to string to ensure clean type matching
alerts_df['stop_id_str'] = alerts_df['stop_id'].astype(str)

# -------------------------------------------------------------------
# 3. Filter Service Alerts by Line 4 Stop IDs
# -----------------------------------------------------------------
line4_stop_alerts = alerts_df[alerts_df['stop_id_str'].isin(line4_stop_ids)].copy()

print(f"\nTotal alerts captured via Line 4 Station Stop IDs: {len(line4_stop_alerts)} rows")

# -------------------------------------------------------------------
# 4. Analyze Disruption Cause Breakdown Across Line 4 Stations
# -------------------------------------------------------------------
# Cause ID breakdown
cause_breakdown = line4_stop_alerts['cause'].value_counts().reset_index()
cause_breakdown.columns = ['Cause_ID', 'Alert_Count']

print("\n--- Cause ID Breakdown at Line 4 Stations ---")
print(cause_breakdown)

# Group alerts by Station Name and Cause ID
# Merge stop names back to alert records for readable station reporting
merged_stop_alerts = line4_stop_alerts.merge(
    stops_df[['stop_id', 'stop_name']],
    left_on='stop_id_str',
    right_on='stop_id',
    how='left'
)

station_cause_summary = (
    merged_stop_alerts.groupby(['stop_name', 'cause'])
    .size()
    .unstack(fill_value=0)
)

print("\n--- Disruption Cause Breakdown by Specific Station ---")
print(station_cause_summary.head(10))

Found 126 stop IDs matching Line 4 stations.

Total alerts captured via Line 4 Station Stop IDs: 58244 rows

--- Cause ID Breakdown at Line 4 Stations ---
   Cause_ID  Alert_Count
0         2        49135
1         3         7213
2        11         1024
3         9          363
4        12          361
5         6          147
6        10            1

--- Disruption Cause Breakdown by Specific Station ---
cause                  2     3   6    9   10   11   12
stop_name                                             
Fridhemsplan         6433  1312  27   30   0  188   65
Gullmarsplan        17192  2405  28  142   0  304  107
Hornstull            4700   770  10   32   0  122   47
Karlaplan            1603   425   0   20   0   54   31
Odenplan             5481   788  10  111   0  142   51
Radiohuset             14     2   0    0   0    6    0
S:t Eriksplan        4855   755   6   13   0  112   46
Stockholm Odenplan   8841   729  66   15   0   94   14
Västerbroplan          16    27   0    

In [5]:
import pandas as pd
import numpy as np
import re

# -------------------------------------------------------------------
# 1. Load Static GTFS Metadata Files
# -------------------------------------------------------------------
# Load stops file (use 'stops.txt' or 'stops.csv' depending on your directory)
stops_df = pd.read_csv('/content/drive/MyDrive/stops.csv', dtype=str, delimiter=';')
routes_df = pd.read_csv('/content/drive/MyDrive/routes.csv', dtype=str, delimiter=';')

# -------------------------------------------------------------------
# 2. Load Service Alerts Parquet Dataset
# -------------------------------------------------------------------
alerts_df = pd.read_parquet('/content/drive/MyDrive/service_alerts_2024.parquet')

# Convert key columns to strings to prevent type mismatch during joins
alerts_df['stop_id_str'] = alerts_df['stop_id'].astype(str)
alerts_df['route_id_str'] = alerts_df['route_id'].astype(str)

# -------------------------------------------------------------------
# 3. Execute 3-Way Merge (Alerts + Stops + Routes)
# -------------------------------------------------------------------
# Step A: Merge with stops.csv for station names & locations
merged_df = alerts_df.merge(
    stops_df[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
    left_on='stop_id_str',
    right_on='stop_id',
    how='left'
)

# Step B: Merge with route.csv for short line names (e.g. "4")
merged_df = merged_df.merge(
    routes_df[['route_id', 'route_short_name', 'route_long_name', 'route_type']],
    left_on='route_id_str',
    right_on='route_id',
    how='left'
)

# Replace unmapped NaN route/stop labels for network-wide alerts
merged_df['route_short_name'] = merged_df['route_short_name'].fillna('Network-Wide (-1)')
merged_df['stop_name'] = merged_df['stop_name'].fillna('Unspecified Stop (-1)')

# -------------------------------------------------------------------
# 4. Engineer Duration, Temporal, and Text Features (TA Advice)
# -------------------------------------------------------------------
# Convert timestamps
merged_df['start'] = pd.to_datetime(merged_df['start'])
merged_df['end'] = pd.to_datetime(merged_df['end'])

# Active alert duration in minutes
merged_df['duration_minutes'] = (
    (merged_df['end'] - merged_df['start']).dt.total_seconds() / 60.0
).astype(np.float32)

# Extract hour and day context
merged_df['start_hour'] = merged_df['start'].dt.hour
merged_df['start_day_of_week'] = merged_df['start'].dt.dayofweek
merged_df['is_weekend'] = (merged_df['start_day_of_week'] >= 5).astype(int)

# Swedish text encoding fixes & string cleaning
encoding_fix = {'Ingen pÃ¥stigning': 'Ingen påstigning'}
merged_df['header_text2'] = merged_df['header_text2'].replace(encoding_fix)
merged_df['description_text2'] = merged_df['description_text2'].replace(encoding_fix)

merged_df['full_text'] = (
    merged_df['header_text2'].fillna('') + ' ' + merged_df['description_text2'].fillna('')
).str.lower()

merged_df['clean_text'] = merged_df['full_text'].apply(lambda t: re.sub(r'[^À-ſ\w\s]', '', t))

# -------------------------------------------------------------------
# 5. Export Enriched Feature Matrix
# -------------------------------------------------------------------
output_cols = [
    'id', 'start', 'end', 'start_hour', 'start_day_of_week', 'is_weekend',
    'duration_minutes', 'cause', 'route_id_str', 'route_short_name',
    'stop_id_str', 'stop_name', 'clean_text'
]

final_alerts_df = merged_df[output_cols].copy()
final_alerts_df.to_parquet('/content/drive/MyDrive/enriched_service_alerts_2024.parquet', index=False)

print(f"Enriched Dataset Created Successfully!")
print(f"Total Rows: {len(final_alerts_df):,}")
print(f"Line 4 Specific Alerts (Short Name '4'): {(final_alerts_df['route_short_name'] == '4').sum():,}")

Enriched Dataset Created Successfully!
Total Rows: 1,972,779
Line 4 Specific Alerts (Short Name '4'): 10


In [9]:
# Load the real-time GTFS data for scheduled arrival information
# This file was previously selected and worked with in cell a9917fcf
# Assuming it contains relevant trip/stop/time information based on common GTFS-RT structures.
rt_scheduled_data = pd.read_csv(
    '/content/gtfs_rt_202403.csv.gz',
    compression='gzip',
    delimiter=';',
    low_memory=False
)

# Select relevant columns and rename them for clarity and consistency.
# Assuming 'route_id', 'stop_id', and 'timestamp' are available in the RT data.
# 'timestamp' is used as a proxy for 'scheduled_arrival_time' for this audit.
# If your RT data has an explicit 'scheduled_arrival_time' column, please adjust.
rt_data_for_merge = rt_scheduled_data[['route_id', 'stop_id', 'timestamp']].copy()
rt_data_for_merge.rename(columns={
    'route_id': 'rt_route_id',
    'stop_id': 'rt_stop_id',
    'timestamp': 'scheduled_arrival_time' # Using 'timestamp' as a proxy for scheduled arrival time
}, inplace=True)

# Convert 'scheduled_arrival_time' to datetime format (assuming Unix timestamp in seconds)
rt_data_for_merge['scheduled_arrival_time'] = pd.to_datetime(rt_data_for_merge['scheduled_arrival_time'], unit='s')

# Convert route and stop IDs to string to ensure clean type matching for merging
rt_data_for_merge['rt_route_id'] = rt_data_for_merge['rt_route_id'].astype(str)
rt_data_for_merge['rt_stop_id'] = rt_data_for_merge['rt_stop_id'].astype(str)

# Merge with the existing `merged_df` (from cell ElUyXsJKBY6F).
# This adds the `scheduled_arrival_time` to our alerts DataFrame.
# We use a left merge to keep all alert records and potentially add scheduled times where available.
global merged_df # Declare merged_df as global to modify the variable in the notebook scope
merged_df = merged_df.merge(
    rt_data_for_merge,
    left_on=['route_id_str', 'stop_id_str'],
    right_on=['rt_route_id', 'rt_stop_id'],
    how='left'
)

# Clean up temporary merge keys if they are not needed afterwards
merged_df.drop(columns=['rt_route_id', 'rt_stop_id'], inplace=True)

# Rename 'start' and 'end' columns to match the audit cell's expected names
merged_df.rename(columns={
    'start': 'alert_start_time',
    'end': 'alert_end_time'
}, inplace=True)

# Derive 'is_alert_active' based on 'alert_start_time', 'alert_end_time', and 'scheduled_arrival_time'.
# An alert is considered active for a given scheduled arrival if the scheduled arrival time
# falls within or exactly on the alert's start and end times.
# Handle cases where 'scheduled_arrival_time' might be NaN due to the left merge (no matching RT data).
merged_df['is_alert_active'] = (
    (merged_df['scheduled_arrival_time'].notna()) & # Ensure we have a scheduled time to compare against
    (merged_df['alert_start_time'] <= merged_df['scheduled_arrival_time']) &
    (merged_df['alert_end_time'] >= merged_df['scheduled_arrival_time'])
).astype(int)

print("Merged scheduled arrival data into merged_df and derived 'is_alert_active' column.")
print(f"New merged_df shape: {merged_df.shape}")
print("Sample of newly added/derived columns:")
display(merged_df[['id', 'alert_start_time', 'alert_end_time', 'scheduled_arrival_time', 'is_alert_active']].head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/gtfs_rt_202403.csv.gz'

In [4]:
import pandas as pd

df = pd.read_parquet('enriched_service_alerts_2024.parquet')

# Station list for Line 4
line4_stations = ['Gullmarsplan', 'Odenplan', 'Stockholm Odenplan', 'Fridhemsplan',
                  'Hornstull', 'S:t Eriksplan', 'Karlaplan', 'Radiohuset', 'Västerbroplan']

# Filter by Line 4 station names
line4_enriched_alerts = df[df['stop_name'].isin(line4_stations)]

print(f"Total Enriched Service Alerts for Line 4 Stations: {len(line4_enriched_alerts):,} rows")
print("\nAlert Breakdown by Station:")
print(line4_enriched_alerts['stop_name'].value_counts())

Total Enriched Service Alerts for Line 4 Stations: 58,244 rows

Alert Breakdown by Station:
stop_name
Gullmarsplan          20178
Stockholm Odenplan     9759
Fridhemsplan           8055
Odenplan               6583
S:t Eriksplan          5787
Hornstull              5681
Karlaplan              2133
Västerbroplan            46
Radiohuset               22
Name: count, dtype: int64


In [3]:
import pandas as pd

# -------------------------------------------------------------------
# 1. FILE PATHS & SELECTION CRITERIA
# -------------------------------------------------------------------
# Path to your Google Drive / local gzipped GTFS-RT file
input_gz_file = '/content/drive/MyDrive/gtfs_rt_202401.csv.gz'
output_parquet = 'line4_gtfs_rt_202401.parquet'

line4_route_id = '9011001000400000'

# -------------------------------------------------------------------
# 2. CHUNKED EXTRACTION (LIGHTWEIGHT RAM USAGE)
# -------------------------------------------------------------------
filtered_chunks = []
chunk_size = 100_000  # Process 100,000 rows at a time

print(f"Reading {input_gz_file} directly in chunks...")

# pandas handles 'compression=gzip' automatically from the file extension
for i, chunk in enumerate(pd.read_csv(input_gz_file, chunksize=chunk_size, low_memory=False, delimiter=';')):
    # Standardize route_id column to string
    chunk['route_id'] = chunk['route_id'].astype(str)

    # Filter rows matching Line 4 Route ID
    line4_chunk = chunk[chunk['route_id'] == line4_route_id]

    if not line4_chunk.empty:
        filtered_chunks.append(line4_chunk)

    if (i + 1) % 10 == 0:
        print(f"Processed {(i + 1) * chunk_size:,} rows...")

# -------------------------------------------------------------------
# 3. COMBINE & EXPORT COMPACT PARQUET
# -------------------------------------------------------------------
if filtered_chunks:
    line4_rt_df = pd.concat(filtered_chunks, ignore_index=True)
    print(f"\nExtraction complete! Total Line 4 GTFS-RT records found: {len(line4_rt_df):,}")

    # Save as compressed Parquet
    line4_rt_df.to_parquet(output_parquet, index=False)
    print(f"Saved lightweight dataset to '{output_parquet}'!")
else:
    print("\nNo matching Line 4 records found. Check route_id formatting.")

Reading /content/drive/MyDrive/gtfs_rt_202401.csv.gz directly in chunks...
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Processed 6,000,000 rows...
Processed 7,000,000 rows...
Processed 8,000,000 rows...
Processed 9,000,000 rows...
Processed 10,000,000 rows...
Processed 11,000,000 rows...
Processed 12,000,000 rows...
Processed 13,000,000 rows...
Processed 14,000,000 rows...
Processed 15,000,000 rows...
Processed 16,000,000 rows...
Processed 17,000,000 rows...
Processed 18,000,000 rows...

Extraction complete! Total Line 4 GTFS-RT records found: 223,077
Saved lightweight dataset to 'line4_gtfs_rt_202401.parquet'!


In [7]:
import pandas as pd

# -------------------------------------------------------------------
# 1. LOAD LIGHTWEIGHT PARQUET DATASETS
# -------------------------------------------------------------------
rt_df = pd.read_parquet('line4_gtfs_rt_202401.parquet')
alerts_df = pd.read_parquet('/content/drive/MyDrive/enriched_service_alerts_2024.parquet')

# Convert timestamps to datetime format
rt_df['scheduled_arrival'] = pd.to_datetime(rt_df['scheduled_arrival_time_sfm'])
alerts_df['start'] = pd.to_datetime(alerts_df['start'])
alerts_df['end'] = pd.to_datetime(alerts_df['end'])

# Ensure stop IDs are strings for clean matching
rt_df['observed_stop_id'] = rt_df['observed_stop_id'].astype(str)
alerts_df['stop_id_str'] = alerts_df['stop_id_str'].astype(str)

# -------------------------------------------------------------------
# 2. MERGE ON STOP ID & FILTER FOR TEMPORALLY VALID ALERTS
# -------------------------------------------------------------------
# Join arrival records with service alerts on Stop ID
merged = pd.merge(
    rt_df,
    alerts_df[['stop_id_str', 'cause', 'duration_minutes', 'start', 'end']],
    left_on='observed_stop_id',
    right_on='stop_id_str',
    how='left'
)

# Enforce strict zero-leakage rule:
# Alert must start BEFORE/AT scheduled arrival and end AFTER/AT scheduled arrival
merged['is_alert_active'] = (
    (merged['start'] <= merged['scheduled_arrival']) &
    (merged['scheduled_arrival'] <= merged['end'])
).astype(int)

# Calculate lead time in minutes (how long disruption was active before bus arrived)
merged['alert_lead_time_mins'] = (
    (merged['scheduled_arrival'] - merged['start']).dt.total_seconds() / 60.0
)
merged.loc[merged['is_alert_active'] == 0, 'alert_lead_time_mins'] = 0

# If no alert was active, set cause to 0 ('No Disruption')
merged['active_cause'] = merged['cause'].where(merged['is_alert_active'] == 1, 0)

# -------------------------------------------------------------------
# 3. VERIFY MERGED FEATURE MATRIX
# -------------------------------------------------------------------
print("--- TEMPORAL ALIGNMENT RESULTS ---")
print(f"Total January Line 4 Bus Arrivals: {len(rt_df):,}")
print(f"Arrivals Operating During Active Disruptions: {merged['is_alert_active'].sum():,}")
print("\nDisruption Cause Breakdown During Active Bus Arrivals:")
print(merged[merged['is_alert_active'] == 1]['active_cause'].value_counts())

# Save final feature matrix for Martin's ML models
merged.to_parquet('line4_january_ml_features.parquet', index=False)
print("\nSaved ready-to-train dataset: 'line4_january_ml_features.parquet'")

--- TEMPORAL ALIGNMENT RESULTS ---
Total January Line 4 Bus Arrivals: 223,077
Arrivals Operating During Active Disruptions: 0

Disruption Cause Breakdown During Active Bus Arrivals:
Series([], Name: count, dtype: int64)

Saved ready-to-train dataset: 'line4_january_ml_features.parquet'
